# 03. IsolationForest, TC와 OP를 함께 보고 조업조건 비교

같은 구역의 온도와 출력을 한 모델에 넣음. 강종별 수준 차이를 Train에서 맞춘 뒤, 같은 강종과 정비 작업 맥락의 과거 점수와 비교함.

**구역별 후보만 표시함. 하나라도 넘으면 행 전체를 1로 만드는 계산은 없음. 기존 LABEL도 바꾸지 않음.**

| 목차 | 확인할 내용 |
| --- | --- |
| 1. 파일과 실행 설정 | DATA_FILE, PLOT_GROUP만 바꾸기 |
| 2. 조업 기록과 기존 제외 구간 | 강종변경 시각, 정비 작업, 팀 판단 |
| 3. TC와 OP를 같은 모델에 넣기 | 구역 묶음, 현재값과 과거 60초 |
| 4. 이전 LOT으로 학습하기 | 고정 9/2/3, 동일 강종, 모든 강종 |
| 5. 같은 조건의 과거 기록과 비교하기 | Train 기준값, 학습하지 못한 조건 보류 |
| 6. 구역별 결과표 | 후보 수, 보류 수, 같은 행끼리 방법 비교 |
| 7. 결과표 바로 아래 TC와 OP 그래프 | PLOT_GROUP 구역의 표와 그림 |
| 8. 저장한 파일과 해석 범위 | 실행 설정, 사후 분석의 한계 |

## 1. 파일과 실행 설정

`data/processed/`의 CSV 이름만 `DATA_FILE`에서 바꿈. `PLOT_GROUP="Z3"`는 그림에서 볼 구역이며 모델 설정을 바꾸지 않음.

조업이벤트, 정비이력, 정기수리캘린더 원본은 `data/` 또는 `data/raw/`에 둠. 결과는 기존 폴더 아래 CSV 이름, 내용 식별값, 실행 시각별로 새로 저장함.

In [ ]:
DATA_FILE = "cal01_base_v1.csv"
PLOT_GROUP = "Z3"
RUN_FIRST_LOT = False

import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from matplotlib import font_manager, rcParams
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from IPython.display import display
from sklearn.ensemble import IsolationForest

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)
files = {
    "base": BASE_CSV,
    "event": "G-02_조업이벤트.csv",
    "maintenance": "G-02_정비이력.csv",
    "repair_plan": "G-01_정기수리캘린더.csv",
}
for key in ["event", "maintenance", "repair_plan"]:
    name = files[key]
    files[key] = os.path.join(ROOT, "data", name)
    if not os.path.isfile(files[key]):
        files[key] = os.path.join(ROOT, "data", "raw", name)

input_hashes = {}
for key, path in files.items():
    with open(path, "rb") as source_file:
        input_hashes[key] = hashlib.sha256(source_file.read()).hexdigest()

RUN_ID = os.path.splitext(DATA_FILE)[0] + "__" + input_hashes["base"][:8] + "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR = os.path.join(PROCESSED_DIR, "sayletskin", "isolation_forest_ab", RUN_ID)
FIG_DIR = os.path.join(ROOT, "figures", "sayletskin", "isolation_forest_ab", RUN_ID)
os.makedirs(OUT_DIR)
os.makedirs(FIG_DIR)

font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

SEED = 42
N_ESTIMATORS = 60
MAX_FIT_ROWS = 12000
MAX_SAMPLES = 256
MAX_GAP_SECONDS = 15
MIN_PAST_ROWS = 10
MIN_REFERENCE_ROWS = 256
MIN_REFERENCE_MINUTES = 5
TRAIN_QUANTILE = 0.95

TRAIN_LOTS = ["240001", "240024", "240037", "240038", "240044", "240058", "240061", "240069", "240091"]
VALID_LOTS = ["240108", "240114"]
TEST_LOTS = ["240117", "240121", "240133"]
LOTS = TRAIN_LOTS + VALID_LOTS + TEST_LOTS
groups = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}
group_titles = {
    "CLN": "세정 CLN-IN",
    "Z1": "건조 Z1",
    "Z2": "건조 Z2",
    "Z3": "가열 Z3",
    "Z4": "가열 Z4와 CP",
    "Z5": "가열 Z5",
    "Z6": "가열 Z6",
    "Z7": "과시효 Z7",
    "Z8": "과시효 Z8",
    "Z9": "과시효 Z9",
    "COOLING": "냉각 OUT1과 OUT2",
}
GRADE_GROUPS = ["Z3", "Z4", "Z5", "Z6", "Z7", "Z8", "Z9", "COOLING"]
SENSORS = []
for group_sensors in groups.values():
    SENSORS.extend(group_sensors)
META = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON", "LABEL"]
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
}
print("입력:", BASE_CSV)
print("결과:", OUT_DIR)
print("그림:", FIG_DIR)

## 2. 조업 기록과 기존 제외 구간

강종변경은 CR1 이벤트를 과거 방향으로 연결함. 직전 강종과 변경 후 경과분은 실제 이벤트에서 가져옴. 임의로 30분이 지나면 안정 운전이라고 나누지 않음.

정비는 CAL01 작업 시작~종료와 겹치는지, 정기수리는 CR1 계획 날짜에 포함되는지 따로 표시함. **작업 기록과 겹침은 설비 정지나 고장 확정이 아님.** 작업 종료시각을 사용한 사후 대조라 실시간 운전상태 판정이라고 부르지 않음.

058 정지, 061 재가동과 114·117의 기존 제외 구간은 유지함. 나머지는 정상생산으로 단정하지 않고 `별도 운전 상태 기록 없음`으로 표시함.

In [ ]:
df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=META + SENSORS, dtype={"LOT_NO": "string", "GRADE": "string"}, low_memory=False)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["EXCLUSION_REASON"] = df["EXCLUSION_REASON"].fillna("")
df["LABEL"] = pd.to_numeric(df["LABEL"], errors="coerce")
df["GRADE"] = df["GRADE"].fillna("미기록")
for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)

events = pd.read_csv(files["event"], encoding=ENC)
events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
events = events.sort_values("EVT_DT")
df = pd.merge_asof(
    df.sort_values("MEAS_DT"),
    events[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]],
    left_on="MEAS_DT", right_on="EVT_DT", direction="backward",
)
df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
df["MINUTES_SINCE_GRADE_CHANGE"] = (df["MEAS_DT"] - df["EVT_DT"]).dt.total_seconds() / 60
df["EVENT_MATCH"] = df["GRADE"].eq(df["AFT_VAL"]).fillna(False) & df["EVT_DT"].notna()

maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
for column in ["STRT_DT", "END_DT"]:
    maintenance[column] = pd.to_datetime(maintenance[column])
df["MAINT_WORK_IDS"] = ""
for work in maintenance.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(work.STRT_DT) & df["MEAS_DT"].lt(work.END_DT)
    df.loc[overlap, "MAINT_WORK_IDS"] += work.WO_NO + "|"
df["MAINT_WORK_IDS"] = df["MAINT_WORK_IDS"].str.rstrip("|")
df["MAINT_OVERLAP"] = df["MAINT_WORK_IDS"].ne("")

repair = pd.read_csv(files["repair_plan"], encoding=ENC)
repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
df["REPAIR_PLAN_IDS"] = ""
for plan in repair.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(plan.STRT_DT) & df["MEAS_DT"].lt(plan.END_EXCLUSIVE)
    df.loc[overlap, "REPAIR_PLAN_IDS"] += plan.PLAN_ID + "|"
df["REPAIR_PLAN_IDS"] = df["REPAIR_PLAN_IDS"].str.rstrip("|")

df["CONTEXT"] = "작업 기록 없음"
df.loc[df["MAINT_OVERLAP"], "CONTEXT"] = "정비 작업 기록과 겹침"
df.loc[df["REPAIR_PLAN_IDS"].ne(""), "CONTEXT"] = "정기수리 계획 기간"
df["OPERATING_NOTE"] = "별도 운전 상태 기록 없음"
df.loc[df["EXCLUDED"], "OPERATING_NOTE"] = df.loc[df["EXCLUDED"], "EXCLUSION_REASON"]
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240058"), "OPERATING_NOTE"] = "정기수리를 위해 온도를 내림, 팀 확정"
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240061"), "OPERATING_NOTE"] = "정기수리 후 재가동하며 온도를 올림, 팀 확정"

CONTEXT_META = ["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL", "MINUTES_SINCE_GRADE_CHANGE", "EVENT_MATCH", "MAINT_OVERLAP", "MAINT_WORK_IDS", "REPAIR_PLAN_IDS", "CONTEXT", "OPERATING_NOTE"]
context_summary = df.groupby(["LOT_NO", "GRADE", "CONTEXT", "OPERATING_NOTE"], dropna=False).agg(
    ROWS=("ROW_ID", "size"), START=("MEAS_DT", "min"), END=("MEAS_DT", "max"),
).reset_index()
context_summary.to_csv(os.path.join(OUT_DIR, "operating_context.csv"), index=False, encoding=ENC)
display(context_summary.rename(columns={
    "LOT_NO": "LOT", "GRADE": "강종", "CONTEXT": "작업 기록과의 관계",
    "OPERATING_NOTE": "팀 판단 또는 상태 기록", "ROWS": "행수",
    "START": "첫 시각", "END": "마지막 시각",
}))

## 3. TC와 OP를 같은 모델에 넣기

Z1~Z8은 같은 번호의 TC와 OP를 함께 입력함. 예를 들어 **Z3 모델은 TC-Z3와 OP-Z3의 현재값, 초당 변화량, 직전 60초 평균과의 차이, 표준편차를 모두 봄.** Z4에는 CP-Z4도 넣음.

CLN-IN과 TC-Z9에는 대응 OP가 없어 각각 따로 봄. 냉각은 TC-OUT1과 TC-OUT2를 묶음.

LOT, 기록 공백 15초, 제외 여부, 강종변경 이벤트, 정비 작업 맥락이 바뀌면 과거 계산을 끊음. 결측은 채우지 않음. 비교에 필요한 입력이 부족하면 그 구역만 보류함.

세정·건조는 강종에 반응하지 않는다는 데이터사전에 따라 강종을 섞어 기준을 계산함. 나머지 구역에는 강종별 기준과 **강종변경 후 경과분의 log(1+분)**을 사용함. TC·OP의 반대 움직임 자체를 이상 규칙으로 넣지 않음.

한 센서의 큰 변화도 구역 점수를 높일 수 있음. 후보가 된 이유는 아래 그래프의 TC·OP 움직임과 조업 기록을 함께 보고 해석함.

In [ ]:
seconds = df["MEAS_DT"].diff().dt.total_seconds()
breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True)
breaks = breaks | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
breaks = breaks | df["EXCLUDED"].ne(df["EXCLUDED"].shift())
breaks = breaks | df["GRADE"].ne(df["GRADE"].shift())
breaks = breaks | df["EVT_ID"].ne(df["EVT_ID"].shift()).fillna(True)
breaks = breaks | df["CONTEXT"].ne(df["CONTEXT"].shift())
df["SEGMENT"] = breaks.cumsum()
features = {}

for sensor in SENSORS:
    feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"])
    for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
        values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
        elapsed = values.index.to_series().diff().dt.total_seconds()
        past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
        feature.loc[block.index, "VALUE"] = values.to_numpy()
        feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
        feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
        feature.loc[block.index, "STD_60S"] = past.std().to_numpy()
    features[sensor] = feature.replace([np.inf, -np.inf], np.nan).astype("float32")

group_inputs = {}
group_valid = {}
for group, sensors in groups.items():
    parts = []
    for sensor in sensors:
        parts.append(features[sensor].add_prefix(sensor + "__"))
    group_input = pd.concat(parts, axis=1)
    if group in GRADE_GROUPS:
        group_input["GRADE_CHANGE_LOG_MIN"] = np.log1p(df["MINUTES_SINCE_GRADE_CHANGE"])
    group_inputs[group] = group_input.astype("float32")
    valid = np.isfinite(group_input.to_numpy()).all(axis=1) & ~df["EXCLUDED"].to_numpy()
    if group in GRADE_GROUPS:
        valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
    group_valid[group] = valid

lot_info = df.groupby("LOT_NO", sort=False).agg(
    GRADE=("GRADE", "first"), START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"), TOTAL_ROWS=("ROW_ID", "size"),
)
group_table = []
for group, sensors in groups.items():
    group_table.append({
        "구역": group_titles[group], "함께 입력하는 센서": ", ".join(sensors),
        "강종 기준": "같은 강종" if group in GRADE_GROUPS else "강종 구분 없음",
        "계산 가능한 행": int(group_valid[group].sum()),
    })
display(pd.DataFrame(group_table))

## 4. 이전 LOT으로 학습하기

고정 9/2/3은 앞 9개를 학습, 108·114를 검증, 117·121·133을 시험으로 둠. 다른 두 방법은 해당 LOT보다 먼저 끝난 동일 강종 또는 모든 강종을 사용함.

이전 LOT이 없으면 생략함. 001 옵션은 실제 관측 시간의 60%, 80%에서 나눔. 학습과 기준 계산에 Valid/Test를 넣지 않음.

In [ ]:
jobs = [{"case": "fixed_9_2_3", "method": "fixed_9_2_3", "roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS}}]

for method in ["same_grade_history", "all_history"]:
    for lot in LOTS:
        history = []
        if lot in lot_info.index:
            previous = lot_info.loc[lot_info["END"].lt(lot_info.loc[lot, "START"])].sort_values("START")
            if method == "same_grade_history":
                previous = previous.loc[previous["GRADE"].eq(lot_info.loc[lot, "GRADE"])]
            history = previous.index.tolist()
        jobs.append({"case": method + "_" + lot, "method": method, "roles": {"train": history, "test": [lot]}})

if RUN_FIRST_LOT and "240001" in lot_info.index:
    first_start = lot_info.loc["240001", "START"]
    first_span = lot_info.loc["240001", "END"] - first_start
    jobs.append({
        "case": "first_lot_60_20_20",
        "method": "first_lot_60_20_20",
        "roles": {"train": ["240001"], "valid": ["240001"], "test": ["240001"]},
        "cuts": [first_start + first_span * 0.6, first_start + first_span * 0.8],
    })

plan_rows = []
for job in jobs:
    job["indices"] = {}
    for role, lots in job["roles"].items():
        selected = df["LOT_NO"].isin(lots)
        if "cuts" in job:
            if role == "train":
                selected = selected & df["MEAS_DT"].lt(job["cuts"][0])
            elif role == "valid":
                selected = selected & df["MEAS_DT"].ge(job["cuts"][0]) & df["MEAS_DT"].lt(job["cuts"][1])
            else:
                selected = selected & df["MEAS_DT"].ge(job["cuts"][1])
        job["indices"][role] = df.index[selected].to_numpy()

    train_index = job["indices"]["train"]
    job["train_index"] = train_index[~df.loc[train_index, "EXCLUDED"].to_numpy()]
    job["status"] = "실행" if len(job["train_index"]) else "생략"
    job["reason"] = ""
    if not job["roles"]["train"]:
        job["reason"] = "이전 동일 강종 기록 없음" if job["method"] == "same_grade_history" else "이전 LOT 기록 없음"
    elif not len(job["train_index"]):
        job["reason"] = "학습에 사용할 제외되지 않은 행 없음"
    if job["method"] == "fixed_9_2_3":
        missing_lots = []
        for required_lot in LOTS:
            if required_lot not in lot_info.index:
                missing_lots.append(required_lot)
        if missing_lots:
            job["train_index"] = np.array([], dtype=int)
            job["status"] = "생략"
            job["reason"] = "고정 9/2/3에 필요한 LOT 없음: " + ",".join(missing_lots)

    for role, lots in job["roles"].items():
        for lot in lots:
            index = job["indices"][role]
            index = index[df.loc[index, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
            n_common = int((~df.loc[index, "EXCLUDED"]).sum())
            status, reason = job["status"], job["reason"]
            if not len(index):
                status, reason = "생략", "CSV에 해당 LOT 또는 시간 구간 없음"
            elif not n_common:
                status, reason = "생략", "계산에 사용할 제외되지 않은 행 없음"
            plan_rows.append({
                "CASE": job["case"],
                "METHOD": job["method"],
                "ROLE": role,
                "LOT_NO": lot,
                "GRADE": lot_info.loc[lot, "GRADE"] if lot in lot_info.index else "",
                "START": df.loc[index, "MEAS_DT"].min(),
                "END": df.loc[index, "MEAS_DT"].max(),
                "STATUS": status,
                "REASON": reason,
                "TOTAL_ROWS": len(index),
                "COMMON_ROWS": n_common,
                "TRAIN_LOTS": ",".join(job["roles"]["train"]),
            })

split_plan = pd.DataFrame(plan_rows)
split_plan.to_csv(os.path.join(OUT_DIR, "split_plan.csv"), index=False, encoding=ENC)
display(split_plan.loc[split_plan["ROLE"].ne("train"), ["METHOD", "ROLE", "LOT_NO", "TRAIN_LOTS", "COMMON_ROWS", "STATUS", "REASON"]].rename(columns={
    "METHOD": "방법",
    "ROLE": "역할",
    "LOT_NO": "LOT",
    "TRAIN_LOTS": "학습 LOT",
    "COMMON_ROWS": "기존 제외를 뺀 행",
    "STATUS": "상태",
    "REASON": "이유",
}))

## 5. 같은 조건의 과거 기록과 비교하기

정비 작업과 겹치는 기록은 겹치지 않는 기록과 **모델을 나눔**. Z3 이후 구역은 Train의 같은 강종에서 중앙값과 가운데 50% 폭을 계산해 입력 수준을 맞춤. 세정·건조는 강종을 나누지 않음.

이렇게 맞춘 TC·OP를 한 모델에 넣고, **같은 강종·같은 작업 맥락의 Train 점수 95% 지점**을 넘는지 확인함. 정상 범위로 확정한 값은 아님.

기준 계산에는 최소 256행과 기록이 있는 1분 칸 5개가 필요함. 계산용 최소 설정이며 충분한 정상 자료라는 뜻은 아님. 같은 조건의 과거 기록이 없으면 판정을 비워 두고 이유를 표시함.

**행 전체 0/1과 센서 후보 합산은 만들지 않음.** 구역별 후보 1도 고장 확정이 아닌, 같은 조건의 과거 기록과 다른 조합이라는 뜻임.

강종 변경 후 경과시간도 입력이므로, 센서값이 비슷해도 그 시간이 과거와 다르면 후보가 될 수 있음. 여러 강종은 입력 수준을 맞춰 같은 모델에 넣으며 강종별 관계를 따로 학습한 것은 아님.

In [ ]:
threshold_rows = []
reference_rows = []
fit_parts = []
prediction_parts = []

for job in jobs:
    evaluation_index = np.array([], dtype=int)
    role_values = []
    for role in ["valid", "test"]:
        if role in job["indices"]:
            index = job["indices"][role]
            evaluation_index = np.concatenate([evaluation_index, index])
            role_values.extend([role] * len(index))

    result = df.loc[evaluation_index, META + CONTEXT_META].copy()
    result["CASE"] = job["case"]
    result["METHOD"] = job["method"]
    result["ROLE"] = role_values
    train_index = job["train_index"]

    for group, group_input in group_inputs.items():
        score_col = group + "__SCORE"
        flag_col = group + "__FLAG"
        status_col = group + "__STATUS"
        result[score_col] = np.nan
        result[flag_col] = np.nan
        result[status_col] = "같은 조업 맥락 없음"
        result.loc[~group_valid[group][evaluation_index], status_col] = "입력 부족"
        if group in GRADE_GROUPS:
            result.loc[~result["EVENT_MATCH"], status_col] = "강종 연결 불일치"
        result.loc[result["EXCLUDED"], status_col] = "기존 제외"

        if not len(train_index):
            mask = ~result["EXCLUDED"] & group_valid[group][evaluation_index]
            result.loc[mask, status_col] = job["reason"] or "이전 LOT 없음"
            continue

        usable_train = train_index[group_valid[group][train_index]]
        usable_eval = evaluation_index[group_valid[group][evaluation_index]]
        grade_key = df["GRADE"] if group in GRADE_GROUPS else pd.Series("강종 구분 없음", index=df.index)

        for context in df.loc[usable_train, "CONTEXT"].unique():
            context_train = usable_train[df.loc[usable_train, "CONTEXT"].eq(context).to_numpy()]
            context_eval = usable_eval[df.loc[usable_eval, "CONTEXT"].eq(context).to_numpy()]
            if not len(context_eval):
                continue
            result.loc[context_eval, status_col] = "같은 강종 기준 없음"
            normalized_train = pd.DataFrame(np.nan, index=context_train, columns=group_input.columns)
            normalized_eval = pd.DataFrame(np.nan, index=context_eval, columns=group_input.columns)
            references = []

            for grade in grade_key.loc[context_train].unique():
                reference_index = context_train[grade_key.loc[context_train].eq(grade).to_numpy()]
                target_index = context_eval[grade_key.loc[context_eval].eq(grade).to_numpy()]
                minute_count = df.loc[reference_index, "MEAS_DT"].dt.floor("min").nunique()
                lot_count = df.loc[reference_index, "LOT_NO"].nunique()
                enough = len(reference_index) >= MIN_REFERENCE_ROWS and minute_count >= MIN_REFERENCE_MINUTES
                if not enough:
                    result.loc[target_index, status_col] = "기준 표본 부족"
                    reference_rows.append({
                        "CASE": job["case"], "GROUP": group, "CONTEXT": context, "GRADE_KEY": grade,
                        "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                        "LOT_COUNT": lot_count, "FEATURE": "", "CENTER": np.nan,
                        "IQR": np.nan, "SCALE": np.nan, "STATUS": "기준 표본 부족",
                    })
                    continue

                values = group_input.loc[reference_index]
                center = values.median()
                iqr = values.quantile(0.75) - values.quantile(0.25)
                scale = iqr.where(iqr.gt(0), values.std(ddof=0))
                scale = scale.where(scale.gt(0), 1.0)
                normalized_train.loc[reference_index] = (values - center) / scale
                normalized_eval.loc[target_index] = (group_input.loc[target_index] - center) / scale
                references.append((grade, reference_index, target_index, minute_count, lot_count))
                for feature_name in group_input.columns:
                    reference_rows.append({
                        "CASE": job["case"], "GROUP": group, "CONTEXT": context, "GRADE_KEY": grade,
                        "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                        "LOT_COUNT": lot_count, "FEATURE": feature_name,
                        "CENTER": float(center[feature_name]), "IQR": float(iqr[feature_name]),
                        "SCALE": float(scale[feature_name]), "STATUS": "기준 계산",
                    })

            normalized_train = normalized_train.dropna().astype("float32")
            if normalized_train.empty:
                continue
            fit_candidates = normalized_train.index.to_numpy()
            train_lots = df.loc[fit_candidates, "LOT_NO"].unique()
            per_lot = MAX_FIT_ROWS // len(train_lots)
            picked = []
            rng = np.random.default_rng(SEED)
            for lot in train_lots:
                index = fit_candidates[df.loc[fit_candidates, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
                picked.extend(rng.choice(index, size=min(per_lot, len(index)), replace=False).tolist())
            fit_index = np.sort(np.array(picked, dtype=int))
            fit_record = df.loc[fit_index, ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "CONTEXT"]].copy()
            fit_record["CASE"] = job["case"]
            fit_record["GROUP"] = group
            fit_parts.append(fit_record)

            model = IsolationForest(n_estimators=N_ESTIMATORS, max_samples=min(MAX_SAMPLES, len(fit_index)), contamination="auto", random_state=SEED, n_jobs=2)
            model.fit(normalized_train.loc[fit_index].to_numpy())
            train_scores = pd.Series(-model.score_samples(normalized_train.to_numpy()), index=normalized_train.index)
            for grade, reference_index, target_index, minute_count, lot_count in references:
                threshold = float(train_scores.loc[reference_index].quantile(TRAIN_QUANTILE))
                threshold_rows.append({
                    "CASE": job["case"], "METHOD": job["method"], "GROUP": group,
                    "CONTEXT": context, "GRADE_KEY": grade, "THRESHOLD": threshold,
                    "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                    "LOT_COUNT": lot_count, "FIT_ROWS": len(fit_index),
                    "TRAIN_LOTS": ",".join(df.loc[reference_index, "LOT_NO"].unique()),
                    "MODEL_TRAIN_LOTS": ",".join(train_lots),
                })
                if len(target_index):
                    score = -model.score_samples(normalized_eval.loc[target_index].to_numpy(dtype="float32"))
                    result.loc[target_index, score_col] = score
                    result.loc[target_index, flag_col] = (score > threshold).astype("int8")
                    result.loc[target_index, status_col] = "계산 완료"

    prediction_parts.append(result)
    print(job["case"], "| 평가 기록", len(result))

predictions = pd.concat(prediction_parts, ignore_index=True)
thresholds = pd.DataFrame(threshold_rows)
references = pd.DataFrame(reference_rows)
fit_rows = pd.concat(fit_parts, ignore_index=True) if fit_parts else pd.DataFrame()
predictions.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False, encoding=ENC, float_format="%.17g")
thresholds.to_csv(os.path.join(OUT_DIR, "thresholds.csv"), index=False, encoding=ENC, float_format="%.17g")
references.to_csv(os.path.join(OUT_DIR, "train_reference.csv"), index=False, encoding=ENC, float_format="%.17g")
fit_rows.to_csv(os.path.join(OUT_DIR, "fit_rows.csv"), index=False, encoding=ENC)

## 6. 구역별 결과표

후보 수와 비율을 구역별로 봄. 구역 하나가 기준을 넘어도 행 전체에 1을 붙이지 않음.

`metrics.csv`의 기존 행 라벨과 겹친 수는 참고용임. 행 라벨은 구역별 정답이 아니므로 Precision, Recall, F1을 계산하지 않음.

117, 121, 133은 각 구역에서 계산 가능한 방법들의 공통 행으로 다시 비교함. 계산하지 못한 이유도 함께 남김.

In [ ]:
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
}
label_by_row = df.set_index("ROW_ID")["LABEL"]
metric_rows = []
for (case, method, role, lot), block in predictions.groupby(["CASE", "METHOD", "ROLE", "LOT_NO"], sort=False):
    labels = block["ROW_ID"].map(label_by_row)
    for group in groups:
        flag = block[group + "__FLAG"]
        pending = block.loc[flag.isna(), group + "__STATUS"].value_counts()
        pending_text = []
        for status, count in pending.items():
            pending_text.append(str(status) + " " + str(count) + "행")

        metric_rows.append({
            "CASE": case,
            "METHOD": method,
            "ROLE": role,
            "LOT_NO": lot,
            "GRADE": block["GRADE"].iloc[0],
            "GROUP": group,
            "RAW_ROWS": len(block),
            "SCORED_ROWS": int(flag.notna().sum()),
            "CANDIDATE_ROWS": int(flag.eq(1).sum()),
            "CANDIDATE_RATE": flag.mean(),
            "PENDING_ROWS": int(flag.isna().sum()),
            "PENDING_REASON": ", ".join(pending_text),
            "ROW_LABEL1_IN_SCORED": int((labels.eq(1) & flag.notna()).sum()),
            "CANDIDATE_AND_ROW_LABEL1": int((labels.eq(1) & flag.eq(1)).sum()),
        })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding=ENC, float_format="%.7g")
show_names = {
    "METHOD": "학습 방법",
    "ROLE": "확인 구간",
    "LOT_NO": "LOT",
    "GROUP": "함께 본 구역",
    "RAW_ROWS": "원본 행",
    "SCORED_ROWS": "판정한 행",
    "CANDIDATE_ROWS": "구역 후보 행",
    "CANDIDATE_RATE": "구역 후보 비율",
    "PENDING_ROWS": "판정 보류 행",
    "PENDING_REASON": "보류 이유",
    "ROW_LABEL1_IN_SCORED": "판정한 행의 기존 라벨 1",
    "CANDIDATE_AND_ROW_LABEL1": "구역 후보와 기존 행 라벨 1이 겹친 수",
}
show_columns = [
    "METHOD", "ROLE", "LOT_NO", "GROUP", "SCORED_ROWS",
    "CANDIDATE_ROWS", "CANDIDATE_RATE", "PENDING_ROWS", "PENDING_REASON",
]
print("각 구역의 판정 결과. 후보 비율의 분모는 그 구역에서 판정한 행임.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(metrics[show_columns].replace({"METHOD": method_names, "GROUP": group_titles}).rename(columns=show_names).round(4))

# 같은 LOT, 같은 구역에서 계산 가능한 방법들의 공통 행만 비교한다.
comparison_rows = []
for lot in TEST_LOTS:
    lot_predictions = predictions.loc[predictions["LOT_NO"].eq(lot) & predictions["ROLE"].eq("test")]
    cases = list(lot_predictions.groupby(["CASE", "METHOD"], sort=False))
    for group in groups:
        common_ids = None
        available_methods = []
        for (case, method), block in cases:
            ids = set(block.loc[block[group + "__FLAG"].notna(), "ROW_ID"])
            if ids:
                available_methods.append(method)
                common_ids = ids if common_ids is None else common_ids.intersection(ids)
        if common_ids is None:
            common_ids = set()

        for (case, method), block in cases:
            flag = block[group + "__FLAG"]
            common = block.loc[block["ROW_ID"].isin(common_ids) & flag.notna()]
            common_flag = common[group + "__FLAG"]
            if not flag.notna().any():
                status = ", ".join(block[group + "__STATUS"].dropna().unique())
            elif len(available_methods) < 2:
                status = "비교 가능한 방법이 1개뿐임"
            elif not len(common):
                status = "방법 사이에 공통으로 판정한 행 없음"
            else:
                status = "같은 행에서 비교"
            comparison_rows.append({
                "CASE": case,
                "METHOD": method,
                "LOT_NO": lot,
                "GROUP": group,
                "COMMON_METHODS": ", ".join(available_methods),
                "N_AVAILABLE_METHODS": len(available_methods),
                "COMMON_ROWS": len(common),
                "CANDIDATE_ROWS": int(common_flag.eq(1).sum()),
                "CANDIDATE_RATE": common_flag.mean(),
                "STATUS": status,
            })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False, encoding=ENC, float_format="%.7g")
comparison_names = dict(show_names)
comparison_names.update({
    "COMMON_ROWS": "방법 간 공통 판정 행",
    "STATUS": "비교 상태",
})
print("117, 121, 133: 구역별로 같은 행에서 방법 비교")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[["LOT_NO", "GROUP", "METHOD", "COMMON_ROWS", "CANDIDATE_ROWS", "CANDIDATE_RATE", "STATUS"]].replace({"METHOD": method_names, "GROUP": group_titles}).rename(columns=comparison_names).round(4))

## 7. 결과표 바로 아래 TC와 OP 그래프

1번의 `PLOT_GROUP`으로 볼 구역을 선택함. 기본 `Z3`은 TC-Z3와 OP-Z3를 같은 시간축에서 봄.

표 아래 노란색은 그 구역의 이상 후보 비율임. 진할수록 1분 동안 판정한 행 중 후보가 많았음. 회색은 판정 보류 또는 관측 없음.

정비 작업, 수리 계획, 기존 제외는 별도 줄로 표시함. 시간 겹침은 원인 확인을 위한 기록이며 고장 또는 정지의 확정 근거가 아님.

In [ ]:
plot_sensors = groups[PLOT_GROUP]
context_colors = {
    "작업 기록 없음": "#f4f6f8",
    "정비 작업 기록과 겹침": "#87b6ce",
    "정기수리 계획 기간": "#bca7cf",
}
candidate_colors = LinearSegmentedColormap.from_list("group_candidate_rate", ["#fffbee", "#f4c24a", "#a56800"])
candidate_colors.set_bad("#d9dde2")

for lot in LOTS:
    lot_predictions = predictions.loc[predictions["LOT_NO"].eq(lot)]
    if lot_predictions.empty:
        continue
    raw = df.loc[df["LOT_NO"].eq(lot)].sort_values("MEAS_DT").copy()
    if raw.empty:
        continue

    print("LOT", lot, "| 함께 볼 구역:", group_titles[PLOT_GROUP])
    grade_events = raw[["EVT_DT", "BEF_VAL", "AFT_VAL"]].drop_duplicates()
    for event in grade_events.itertuples(index=False):
        if pd.notna(event.EVT_DT):
            print("이 관측보다 앞선 강종 변경 기록:", event.EVT_DT, "|", event.BEF_VAL, "→", event.AFT_VAL)
        else:
            print("앞선 강종 변경 기록을 연결하지 못함.")
    print("운전 상태 메모:", " / ".join(raw["OPERATING_NOTE"].dropna().astype(str).unique()))
    print("정비 표시는 작업 시간의 겹침임. 고장 또는 정지 판정이 아님.")

    lot_metrics = metrics.loc[metrics["LOT_NO"].eq(lot) & metrics["GROUP"].eq(PLOT_GROUP), show_columns]
    display(lot_metrics.replace({"METHOD": method_names, "GROUP": group_titles}).rename(columns=show_names).round(4))

    # 빈 시간은 이어 그리지 않고, 센서 결측도 그대로 끊는다.
    gap = raw["MEAS_DT"].diff().dt.total_seconds()
    raw["DRAW_SEGMENT"] = (gap.gt(MAX_GAP_SECONDS) | gap.le(0)).cumsum()
    methods = []
    for method in method_names:
        if lot_predictions["METHOD"].eq(method).any():
            methods.append(method)
    n_sensors = len(plot_sensors)
    heights = [2] * n_sensors + [0.9, 1.8]
    fig, axes = plt.subplots(
        n_sensors + 2, 1,
        figsize=(14, 2.1 * n_sensors + 3.8),
        sharex=True,
        gridspec_kw={"height_ratios": heights},
    )
    for axis, sensor in zip(axes[:n_sensors], plot_sensors):
        for segment, block in raw.groupby("DRAW_SEGMENT", sort=False):
            axis.plot(block["MEAS_DT"], block[sensor], color="#396788", linewidth=0.7)
        unit = "기록값"
        if sensor.startswith("TC-"):
            unit = "℃"
        elif sensor.startswith("OP-"):
            unit = "%"
        axis.set_ylabel(sensor + "\n(" + unit + ")")
        axis.grid(alpha=0.15)

    # 조업 맥락과 기존 제외는 모델 결과와 다른 줄에 표시한다.
    context_axis = axes[-2]
    context_break = raw["CONTEXT"].ne(raw["CONTEXT"].shift()) | raw["DRAW_SEGMENT"].ne(raw["DRAW_SEGMENT"].shift())
    for segment, block in raw.groupby(context_break.cumsum(), sort=False):
        start = block["MEAS_DT"].iloc[0]
        end = block["MEAS_DT"].iloc[-1] + pd.Timedelta(seconds=1)
        color = context_colors.get(block["CONTEXT"].iloc[0], "#f4f6f8")
        context_axis.fill_between([start, end], -0.35, 0.35, color=color)

    exclude_break = raw["EXCLUDED"].ne(raw["EXCLUDED"].shift()) | raw["DRAW_SEGMENT"].ne(raw["DRAW_SEGMENT"].shift())
    for segment, block in raw.groupby(exclude_break.cumsum(), sort=False):
        if block["EXCLUDED"].iloc[0]:
            start = block["MEAS_DT"].iloc[0]
            end = block["MEAS_DT"].iloc[-1] + pd.Timedelta(seconds=1)
            context_axis.fill_between([start, end], 0.65, 1.35, color="#aeb5bd")
            for axis in axes[:n_sensors]:
                axis.axvspan(start, end, color="#c4c7cc", alpha=0.25)
    context_axis.set_yticks([0, 1], ["정비 / 계획 기록", "기존 제외"])
    context_axis.set_ylim(1.5, -0.5)
    context_axis.grid(False)

    minute_edges = pd.date_range(
        raw["MEAS_DT"].min().floor("min"),
        raw["MEAS_DT"].max().floor("min") + pd.Timedelta(minutes=1),
        freq="min",
    )
    minute_index = minute_edges[:-1]
    minute_rates = []
    tick_labels = []
    for method in methods:
        block = lot_predictions.loc[lot_predictions["METHOD"].eq(method)]
        scored = block.loc[block[PLOT_GROUP + "__FLAG"].notna()]
        rate = scored.groupby(scored["MEAS_DT"].dt.floor("min"))[PLOT_GROUP + "__FLAG"].mean()
        minute_rates.append(rate.reindex(minute_index).to_numpy())
        suffix = " (판정 보류)" if scored.empty else ""
        tick_labels.append(method_names[method] + suffix)
    heat = axes[-1].pcolormesh(
        mdates.date2num(minute_edges),
        np.arange(len(tick_labels) + 1) - 0.5,
        np.array(minute_rates),
        cmap=candidate_colors,
        vmin=0,
        vmax=1,
        shading="flat",
    )
    axes[-1].set_yticks(range(len(tick_labels)), tick_labels)
    axes[-1].set_ylim(len(tick_labels) - 0.5, -0.5)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[-1].set_xlabel(str(raw["MEAS_DT"].iloc[0].year) + "년 실제 날짜와 시각")
    axes[-1].set_xlim(minute_edges[0], minute_edges[-1])
    fig.suptitle(DATA_FILE + " | LOT " + lot + " | " + group_titles[PLOT_GROUP] + " 관계와 이상 후보")
    fig.subplots_adjust(left=0.18, right=0.985, top=0.93, bottom=0.23, hspace=0.15)
    color_axis = fig.add_axes([0.38, 0.12, 0.5, 0.018])
    colorbar = fig.colorbar(heat, cax=color_axis, orientation="horizontal", ticks=[0, 0.25, 0.5, 0.75, 1])
    colorbar.ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
    colorbar.set_label("1분 동안 판정한 행 중 이 구역이 이상 후보인 비율")
    fig.text(0.18, 0.035, "정비 / 계획 줄: 하늘색 = 정비 작업과 겹침, 보라색 = 수리 계획 기간, 연한 회색 = 작업 기록 없음", fontsize=8.5)
    fig.text(0.18, 0.01, "후보 그림의 회색 = 판정 보류 또는 관측 없음. 정비 기록의 겹침만으로 고장이나 정지를 뜻하지 않음.", fontsize=8.5)
    fig.savefig(os.path.join(FIG_DIR, "if_compare_" + lot + "_" + PLOT_GROUP + ".png"), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 8. 저장한 파일과 해석 범위

- `predictions.csv`: 시각, 조업 기록, 구역별 점수·후보·보류 이유. 행 전체 판정 없음.
- `metrics.csv`, `comparison.csv`: 구역별 결과, 방법 사이에 같은 행만 비교한 결과.
- `thresholds.csv`, `train_reference.csv`, `fit_rows.csv`: Train 기준과 실제 학습 표본.
- `operating_context.csv`, `split_plan.csv`, `settings.json`: 조업 맥락, 분할, 실행 설정.

강종 기준이 없는 117은 해당 구역을 보류함. 정비 작업과 겹치는데 같은 맥락의 학습이 없을 때도 보류함. 보류는 정상 0이 아님.

기존 LABEL은 행에 붙은 후보라 구역별 정답으로 쓰지 않음. Precision·Recall·F1은 계산하지 않음. 정비 종료시각과 팀 제외 판단을 사용한 사후 분석이며 실시간 고장 예측 성능으로 읽지 않음.

TC·OP의 동시 조합을 함께 보지만, 인과관계나 강판 품질을 확인한 것은 아님. 실제 목표온도·강판 온도·통과속도는 입력에 없음. 다른 존 사이의 전달 지연도 이번 모델 범위에 없음.

In [ ]:
settings = {
    "model_version": "joint_groups_conditioned_v2",
    "input_file": DATA_FILE,
    "input_sha256": input_hashes,
    "run_id": RUN_ID,
    "sklearn_version": sklearn.__version__,
    "seed": SEED,
    "n_estimators": N_ESTIMATORS,
    "max_fit_rows_per_group_context": MAX_FIT_ROWS,
    "max_samples": MAX_SAMPLES,
    "train_quantile": TRAIN_QUANTILE,
    "groups": groups,
    "grade_groups": GRADE_GROUPS,
    "normalization": "Train context/grade median and IQR; zero IQR falls back to Train std then 1",
    "context_models": "separate model for each maintenance/repair context",
    "grade_change_input": "log1p(minutes since past actual CR1 grade-change event), grade-sensitive groups only",
    "reference_min_rows": MIN_REFERENCE_ROWS,
    "reference_min_observed_minutes": MIN_REFERENCE_MINUTES,
    "threshold": "Train same context/grade score quantile, strict greater-than",
    "no_row_level_prediction": True,
    "label_use": "reference overlap counts only, no group ground truth, no precision/recall/F1",
    "rolling_seconds": 60,
    "rolling_min_past_rows": MIN_PAST_ROWS,
    "rolling_closed": "left",
    "rolling_std_ddof": 1,
    "gap_reset_seconds": MAX_GAP_SECONDS,
    "validity": "group-specific finite inputs, excludes existing EXCLUDED, unmatched grade events withheld",
    "fit_sampling": "same quota per available Train LOT, short LOT uses all available rows",
    "fixed_roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS},
    "run_first_lot": RUN_FIRST_LOT,
    "first_lot_policy": "actual observation time 60/20/20, train cutoff exclusive",
    "plot_group": PLOT_GROUP,
    "plot_aggregation": "per-minute fraction of group candidates among computed source rows",
    "operating_context_limit": "work intervals and team exclusions are retrospective; no explicit running/stopped command logs",
}
with open(os.path.join(OUT_DIR, "settings.json"), "w", encoding="utf-8") as settings_file:
    json.dump(settings, settings_file, ensure_ascii=False, indent=2)
print("저장 완료:", OUT_DIR)
print("그림:", FIG_DIR)